
# BPY518 Lecture 2: Image Preprocessing and Noise Reduction
Instructor: Josh Shaevitz  
Email: shaevitz@princeton.edu



## Lecture Outline
- Bit depth, dynamic range, and contrast
- Sources of image noise
- Mean, median, and Gaussian filtering
- Filtering as convolution
- Background subtraction
- Temporal median background estimation



## Why preprocess images?

In Lecture 31 we treated an image as an array of numbers. Real microscopy images are usually not ready for analysis straight out of the camera. Before we threshold, segment, or track anything, we often need to think about:

- how the camera digitized the intensity values,
- whether the image uses the available intensity range well,
- what kind of noise is present,
- and whether a slowly varying background should be removed.

In this lecture we will focus on **single-channel grayscale images**.


In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage

plt.rcParams['figure.dpi'] = 120
plt.rcParams['image.cmap'] = 'gray'



## Image Bit Depth

Bit depth tells us how many distinct intensity levels the camera can represent. An `N`-bit image has `2**N` possible levels.

- 8-bit: 256 gray levels
- 12-bit: 4096 gray levels
- 16-bit: 65,536 gray levels

The figure below keeps the visual comparison from the original slide, but the explanation belongs here in markdown rather than inside the image.

![Bit depth examples](media/Lecture_2/bit_depth_examples_cropped.png)


In [ ]:

# Build a smooth grayscale ramp and quantize it to different bit depths.
gradient = np.tile(np.linspace(0, 1, 256), (64, 1))


def quantize_to_n_bits(image, n_bits):
    levels = 2 ** n_bits - 1
    return np.round(image * levels) / levels


fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, n_bits in zip(axes, [2, 4, 8, 16]):
    quantized = quantize_to_n_bits(gradient, n_bits)
    ax.imshow(quantized, vmin=0, vmax=1, aspect='auto')
    ax.set_title(f'{n_bits}-bit')
    ax.axis('off')

plt.tight_layout()
plt.show()



## Dynamic Range and Contrast

**Dynamic range** is the span between the darkest and brightest intensities an image can represent. **Contrast** is about how much of that range the image actually uses.

- A high-contrast image has bright whites and dark blacks with strong edge detail.
- A low-contrast image looks flat because the pixel values occupy only a narrow range.
- High dynamic range and low contrast can happen at the same time if the available range is large but the measured values cluster together.
- Contrast stretching can make an image easier to see, but it does **not** add new information.


In [ ]:

# Simulate an image that only uses a narrow part of the available range.
low_contrast = 0.45 + 0.12 * gradient
stretched = (low_contrast - low_contrast.min()) / (low_contrast.max() - low_contrast.min())

fig, axes = plt.subplots(2, 2, figsize=(10, 5))
axes[0, 0].imshow(low_contrast, vmin=0, vmax=1, aspect='auto')
axes[0, 0].set_title('Low-contrast image')
axes[0, 0].axis('off')

axes[0, 1].imshow(stretched, vmin=0, vmax=1, aspect='auto')
axes[0, 1].set_title('After contrast stretching')
axes[0, 1].axis('off')

axes[1, 0].hist((255 * low_contrast).ravel(), bins=32, color='dimgray')
axes[1, 0].set_title('Original histogram')
axes[1, 0].set_xlabel('Pixel value')
axes[1, 0].set_ylabel('Count')

axes[1, 1].hist((255 * stretched).ravel(), bins=32, color='steelblue')
axes[1, 1].set_title('Stretched histogram')
axes[1, 1].set_xlabel('Pixel value')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.show()



## Noisy Images

Microscopy images are noisy for physical reasons, not because the microscope is "bad."

- **Shot noise** comes from the randomness of photon arrival.
- **Dark current** is signal generated by the detector even without incoming light.
- **Read noise** comes from electronics during amplification and digitization.

The figure below shows how different noise sources change both the image and its intensity histogram.

![Noisy image examples](media/Lecture_2/noisy_images_cropped.png)


In [ ]:

# Build a simple synthetic image and add Poisson-like and Gaussian noise.
rng = np.random.default_rng(2)
yy, xx = np.mgrid[:128, :128]
clean_image = (
    20
    + 90 * np.exp(-((xx - 42) ** 2 + (yy - 48) ** 2) / (2 * 10**2))
    + 70 * np.exp(-((xx - 88) ** 2 + (yy - 78) ** 2) / (2 * 14**2))
)
shot_noisy = rng.poisson(clean_image).astype(float)
noisy_image = shot_noisy + rng.normal(0, 6, clean_image.shape)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, data, title in zip(
    axes,
    [clean_image, shot_noisy, noisy_image],
    ['Clean synthetic image', 'After shot noise', 'After shot + read noise'],
):
    ax.imshow(data)
    ax.set_title(title)
    ax.axis('off')

plt.tight_layout()
plt.show()



## Dealing with Noise

There are two common strategies:

- If you are fitting a model directly to the data, you may leave the noise in place and let the model absorb it.
- If the noise makes visualization or downstream analysis difficult, you can smooth the image.

The source slides show OpenCV snippets such as `cv2.blur`, `cv2.medianBlur`, and `cv2.GaussianBlur`. In this notebook we will use `scipy.ndimage` for the built-in versions so we do not introduce a new library yet.



## Mean Filtering

A mean filter replaces each pixel with the average value in a local neighborhood.

- Good for random pixel-scale noise
- Easy to understand and implement
- Tends to blur edges

![Mean filtering examples](media/Lecture_2/mean_filter_examples_cropped.png)


In [ ]:

# Reconstruct the slide's brute-force 3x3 mean filter in clean Python.
def mean_filter_brute_force(image):
    image = image.astype(float)
    H, W = image.shape
    filtered = np.zeros((H - 2, W - 2), dtype=float)

    for i in range(1, H - 1):
        for j in range(1, W - 1):
            window = image[i - 1:i + 2, j - 1:j + 2]
            filtered[i - 1, j - 1] = np.mean(window)

    return filtered


base = np.zeros((80, 80), dtype=float)
base[20:60, 25:55] = 1.0
rng = np.random.default_rng(0)
noisy_square = base + 0.25 * rng.normal(size=base.shape)

brute_mean = mean_filter_brute_force(noisy_square)
library_mean = ndimage.uniform_filter(noisy_square, size=3, mode='nearest')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(noisy_square)
axes[0].set_title('Noisy input')
axes[0].axis('off')

axes[1].imshow(brute_mean)
axes[1].set_title('Brute-force 3x3 mean')
axes[1].axis('off')

axes[2].imshow(library_mean[1:-1, 1:-1])
axes[2].set_title('ndimage.uniform_filter')
axes[2].axis('off')

plt.tight_layout()
plt.show()



## Median Filtering

A median filter replaces the center pixel with the median value in a local neighborhood.

- Very useful for isolated outliers such as hot pixels or salt-and-pepper noise
- Often preserves edges better than the mean filter
- Computationally slower because it requires sorting values in the window

![Median filtering examples](media/Lecture_2/median_filter_examples_cropped.png)


In [ ]:

# Median filters shine when a few pixels are wildly wrong.
def add_salt_and_pepper(image, amount=0.05, seed=0):
    rng = np.random.default_rng(seed)
    noisy = image.copy()
    n = int(amount * image.size)

    salt_rows = rng.integers(0, image.shape[0], size=n)
    salt_cols = rng.integers(0, image.shape[1], size=n)
    pepper_rows = rng.integers(0, image.shape[0], size=n)
    pepper_cols = rng.integers(0, image.shape[1], size=n)

    noisy[salt_rows, salt_cols] = 1.0
    noisy[pepper_rows, pepper_cols] = 0.0
    return noisy


clean_shape = np.zeros((100, 100), dtype=float)
clean_shape[20:80, 20:80] = 1.0
clean_shape[40:60, 40:60] = 0.3
salt_pepper = add_salt_and_pepper(clean_shape, amount=0.05, seed=1)

mean_filtered = ndimage.uniform_filter(salt_pepper, size=5, mode='nearest')
median_filtered = ndimage.median_filter(salt_pepper, size=5, mode='nearest')

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, data, title in zip(
    axes,
    [clean_shape, salt_pepper, mean_filtered, median_filtered],
    ['Clean image', 'Salt & pepper noise', 'Mean filter (5x5)', 'Median filter (5x5)'],
):
    ax.imshow(data, vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis('off')

plt.tight_layout()
plt.show()



## Gaussian Filtering

A Gaussian filter computes a weighted average where nearby pixels matter more than distant ones.

- Nearby pixels receive large weights.
- Distant pixels receive small weights.
- The result is usually smoother than the raw image but less edge-blurring than a plain mean filter.
- Large `sigma` values suppress fine structure.

In equation form,

$$
I'(x,y) = \sum_{i=-1}^{1} \sum_{j=-1}^{1} G(i,j)\,I(x+i, y+j)
$$

with Gaussian weights

$$
G(i,j) = \frac{1}{2\pi\sigma^2} \exp\left(-\frac{i^2 + j^2}{2\sigma^2}\right).
$$


In [ ]:

# Adapt the Gaussian-filter code snippet from the slide and compare it to SciPy.
def gaussian(di, dj, sigma):
    return (1 / (2 * math.pi * sigma**2)) * math.exp(-(di**2 + dj**2) / (2 * sigma**2))



def gaussian_filter_brute_force(image, sigma=1.0):
    image = image.astype(float)
    H, W = image.shape
    filtered = np.zeros((H - 2, W - 2), dtype=float)

    for i in range(1, H - 1):
        for j in range(1, W - 1):
            weighted_sum = 0.0
            total_weight = 0.0
            for di in range(-1, 2):
                for dj in range(-1, 2):
                    weight = gaussian(di, dj, sigma)
                    weighted_sum += weight * image[i + di, j + dj]
                    total_weight += weight
            filtered[i - 1, j - 1] = weighted_sum / total_weight

    return filtered


edge_image = np.zeros((80, 80), dtype=float)
edge_image[:, :40] = 0.2
edge_image[:, 40:] = 1.0
rng = np.random.default_rng(4)
noisy_edge = edge_image + 0.15 * rng.normal(size=edge_image.shape)

brute_gaussian = gaussian_filter_brute_force(noisy_edge, sigma=1.0)
library_gaussian = ndimage.gaussian_filter(noisy_edge, sigma=1.0, mode='nearest')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(noisy_edge, vmin=0, vmax=1)
axes[0].set_title('Noisy edge image')
axes[0].axis('off')

axes[1].imshow(brute_gaussian, vmin=0, vmax=1)
axes[1].set_title('Brute-force Gaussian')
axes[1].axis('off')

axes[2].imshow(library_gaussian[1:-1, 1:-1], vmin=0, vmax=1)
axes[2].set_title('ndimage.gaussian_filter')
axes[2].axis('off')

plt.tight_layout()
plt.show()



## Filtering as Convolution

All of these local filters can be written as a **convolution** between the image and a kernel.

For a 3x3 mean filter, the kernel is

$$
M = \frac{1}{9}
\begin{bmatrix}
1 & 1 & 1 \\
1 & 1 & 1 \\
1 & 1 & 1
\end{bmatrix}
$$

and the filtered image is

$$
I'(x,y) = \sum_{i=-1}^{1} \sum_{j=-1}^{1} M(i,j)\,I(x+i, y+j).
$$

In this lecture we use that idea informally: slide a small window across the image, combine the neighboring pixels, and write out a new image. A later lecture will return to convolutions more formally.



## Background Subtraction

Noise is not the only problem in microscopy. We also often have a slowly varying **background** caused by uneven illumination, autofluorescence, scattered light, or non-specific staining.

- If the background is approximately constant, we can subtract a scalar.
- If the background changes across the field of view, we need to estimate a full background image.
- After subtraction, clip negative values so they do not wrap around in integer images.
- A common practical trick is to estimate the background with a very broad blur and subtract that smooth image.

![Types of background subtraction](media/Lecture_2/background_types_cropped.png)

A slide code snippet expressed the same idea as:

```python
background = gaussian_blur(image, large_scale)
image_subtracted = image - background
```


In [ ]:

# Compare constant background subtraction to subtraction of a smooth estimated background.
rng = np.random.default_rng(5)
yy, xx = np.mgrid[:160, :160]
background = 30 + 0.18 * xx + 0.12 * yy
signal = (
    80 * np.exp(-((xx - 45) ** 2 + (yy - 55) ** 2) / (2 * 8**2))
    + 110 * np.exp(-((xx - 110) ** 2 + (yy - 100) ** 2) / (2 * 12**2))
)
image_with_background = background + signal + rng.normal(0, 3, background.shape)

constant_background = np.median(image_with_background[:20, :20])
constant_subtracted = np.clip(image_with_background - constant_background, 0, None)

smooth_background = ndimage.gaussian_filter(image_with_background, sigma=20)
background_subtracted = np.clip(image_with_background - smooth_background, 0, None)

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, data, title in zip(
    axes,
    [image_with_background, constant_subtracted, smooth_background, background_subtracted],
    ['Original image', 'Subtract constant', 'Estimated background', 'Subtract smooth background'],
):
    ax.imshow(data)
    ax.set_title(title)
    ax.axis('off')

plt.tight_layout()
plt.show()



## Temporal Median Background Estimation

For a time-lapse movie, a very useful trick is to compute the median image across time.

```python
median_background = np.median(stack, axis=0)
corrected = stack - median_background
```

This works best when the foreground objects move around and do not spend most of the movie in the same place.

![Temporal median background estimation](media/Lecture_2/temporal_median_background_cropped.png)


In [ ]:

# Build a simple movie with one moving bright object on top of a static background.
rng = np.random.default_rng(6)
yy, xx = np.mgrid[:96, :96]
static_background = 25 + 0.10 * xx + 0.05 * yy
stack = []

for t in range(15):
    cx = 15 + 4 * t
    cy = 48 + 10 * np.sin(t / 2)
    moving_spot = 70 * np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * 5**2))
    frame = static_background + moving_spot + rng.normal(0, 2, static_background.shape)
    stack.append(frame)

stack = np.stack(stack)
median_background = np.median(stack, axis=0)
corrected = np.clip(stack - median_background, 0, None)
frame_id = 7

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, data, title in zip(
    axes,
    [stack[frame_id], median_background, corrected[frame_id]],
    ['Original frame', 'Median background', 'Background-subtracted frame'],
):
    ax.imshow(data)
    ax.set_title(title)
    ax.axis('off')

plt.tight_layout()
plt.show()



## Key Takeaways

- Bit depth controls how many intensity levels are available.
- Dynamic range and contrast are related but not identical.
- Mean, median, and Gaussian filters suppress different kinds of noise and have different tradeoffs.
- Local filtering can be understood as convolution with a small kernel.
- Background subtraction removes slowly varying signal that is not part of the object of interest.
- For movies, a temporal median can estimate the static background surprisingly well.
